# Gas Turbine · помогает ли история датчиков оценивать NOx и CO?

В каждой годовой таблице берём шесть соседних строк со сдвигом на одну строку. Девять датчиков из строк от `t−5` до `t` образуют вход, выброс в строке `t` служит целью. CO и NOx исключены из входов. В файлах нет точных временных меток, поэтому длину окна измеряем в наблюдениях. Для каждого года первые пять строк дают контекст следующим строкам.

| Модель | Что получает | Архитектура | Параметры |
|---|---|---|---:|
| GRU | 6 × 9 | один слой, скрытое состояние 22, линейный выход | 2 201 |
| Оконный MLP | те же 54 значения в том же порядке | два ReLU-слоя ширины 26 | 2 159 |
| Обычный MLP | последние 9 значений | два ReLU-слоя ширины 41 | 2 174 |
| Гиперграфовая MGS-KAN | последние 9 значений | 9 ядер на признак, 5 на пару, 1 на тройку | 2 173 |

Обучение и стандартизация по 2011–2012 годам. Adam, шаг 0,001, батч 64, 100 эпох. Лучшая эпоха по RMSE на 2013 году. Тест 2014–2015 годов оцениваем на концах одних и тех же окон у всех моделей. Сначала проверяем наличие 12 готовых точечных моделей в MLflow. Затем обучаем до 12 оконных моделей, по три пары seed на цель и архитектуру. Выбери GPU и выполни **Run All**.

In [ ]:
PROJECT_DIR = None
DEVICE = "cuda:1"
GPU_BUDGET_MIB = 4096
VERSION = "gas-windows-v1"

In [ ]:
import sys
from pathlib import Path

if PROJECT_DIR is None:
    cwd = Path.cwd().resolve()
    checkout = next(
        (folder for folder in (cwd, *cwd.parents) if (folder / "src/experiments/runner.py").is_file()),
        None,
    )
else:
    checkout = Path(PROJECT_DIR).expanduser()
    if not checkout.is_absolute() or not (checkout / "src/experiments/runner.py").is_file():
        raise ValueError("Укажи в PROJECT_DIR абсолютный путь к корню mgs-kan на сервере.")
    checkout = checkout.resolve()

if checkout is not None and sys.path[0:1] != [str(checkout)]:
    sys.path.insert(0, str(checkout))

try:
    from src.paths import DATA_DIR, ENV_FILE, PROJECT_ROOT
except ModuleNotFoundError as error:
    raise ImportError("Укажи PROJECT_DIR и выполни ячейки заново.") from error

if checkout is not None and PROJECT_ROOT != checkout:
    raise RuntimeError("Ядро уже загрузило src из другой папки. Перезапусти ядро.")
if not (PROJECT_ROOT / "notebooks/baselines/run_series.ipynb").is_file():
    raise RuntimeError("Укажи PROJECT_DIR с рабочей копией mgs-kan и перезапусти ядро.")
print(f"Проект: {PROJECT_ROOT}\nНастройки: {ENV_FILE}\nДанные: {DATA_DIR}")

In [ ]:
from dataclasses import replace
from itertools import product
from math import isclose

import pandas as pd
import torch
from IPython.display import display

from src.config import ModelSettings, MonitorSettings, ResourceSettings, TrackingSettings
from src.data import load_dataset
from src.experiments.search import SearchStage, ValidationGrid
from src.models import make_model, parameter_count
from src.tracking import MlflowTracker
from src.training import Trainer

targets = ("gas_turbine_nox", "gas_turbine_co")
seeds = ((11, 101), (22, 202), (33, 303))
baseline = SearchStage(name="point", models=("mlp", "hypergraph_mgs_kan"), history_only=True,
                       settings=ModelSettings(mlp_width=41, single_kernels=9, pair_kernels=5))
sequence = SearchStage(name="window6", models=("gru_window6", "mlp_window6"),
                       settings=ModelSettings(deep_width=22, mlp_width=26))
window_names = tuple(f"{target}_window6" for target in targets)
point_grid = ValidationGrid(datasets=targets, stages=(baseline,), seeds=seeds, reuse_matching_validation=True)
window_grid = ValidationGrid(datasets=window_names, stages=(sequence,), seeds=seeds, reuse_matching_validation=True)
data = {name: load_dataset(name, root=DATA_DIR, fold=0) for name in (*targets, *window_names)}

tracking = TrackingSettings(_env_file=ENV_FILE)
resources = ResourceSettings(_env_file=ENV_FILE, max_parallel_trials=1, cpu_threads=1,
                             gpu_budget_mib=GPU_BUDGET_MIB)
runtime = dict(version=VERSION, device=DEVICE, root=DATA_DIR, tracking=tracking, resources=resources,
               monitoring=MonitorSettings(_env_file=ENV_FILE))
output = PROJECT_ROOT / "results" / "gas-turbine-windows" / VERSION
for target in targets:
    windows = data[f"{target}_window6"]
    print(f"{target}: {len(windows.train)} окон обучения, {len(windows.val)} валидации, {len(windows.test)} теста")

In [ ]:
point_runs = point_grid.run(**runtime)
window_runs = window_grid.run(**runtime)
runs = pd.concat([point_runs, window_runs], ignore_index=True)
runs["target"] = runs.dataset.str.removesuffix("_window6")
expected = set(product(targets, (*baseline.models, *sequence.models), seeds))
observed = {(r.target, r.model, (r.init_seed, r.shuffle_seed)) for r in runs.itertuples()}
if observed != expected or len(runs) != len(expected) or runs.run_id.nunique() != len(expected):
    raise ValueError("Expected 24 distinct, seed-paired checkpoints")
print(f"Точечные модели из MLflow: {len(point_runs)}. Новых оконных обучений: {(~window_runs.reused).sum()}.")

## Проверка чекпоинтов и оценка

Восстанавливаем лучшую эпоху каждого запуска и сверяем RMSE с MLflow на его исходной валидации. На тесте оконные модели получают весь отрезок, точечные — последнюю строку из того же тензора. В обоих случаях используем один и тот же вектор целей в исходных единицах.

In [ ]:
torch.set_num_threads(resources.cpu_threads)
tracker = MlflowTracker(tracking)
scores = []
for row in runs.itertuples(index=False):
    original = data[row.dataset]
    windows = data[f"{row.target}_window6"]
    point_model = row.model in baseline.models
    stage = baseline if point_model else sequence
    opts = stage.settings.model_copy(update={"features": original.n_features, "init_seed": row.init_seed})
    model = make_model(row.model, opts)
    if parameter_count(model) != row.parameters:
        raise ValueError(f"Parameter count mismatch for {row.run_id}")
    trainer = Trainer(model, tracker=tracker)
    if trainer.load_checkpoint(run_id=row.run_id) != row.best_epoch:
        raise ValueError(f"Checkpoint epoch mismatch for {row.run_id}")
    val_rmse = trainer.evaluate(replace(original, test=original.val))
    if not isclose(val_rmse, row.val_rmse, rel_tol=1e-4, abs_tol=1e-4):
        raise ValueError(f"Validation RMSE mismatch for {row.run_id}")
    test = windows.test_endpoints() if point_model else windows.test
    rmse = trainer.evaluate(replace(windows, test=test))
    scores.append(dict(row._asdict(), n_test=len(test), test_rmse=rmse))
test_runs = pd.DataFrame(scores)

## Сравнение на тесте

Считаем RMSE отдельно для CO и NOx. В сводке — среднее и стандартное отклонение по трём seed. В парных разностях отрицательное значение означает меньшую ошибку у первой модели. Оконный MLP против обычного MLP показывает эффект добавления истории при близком числе параметров. GRU против оконного MLP сравнивает архитектуры при одинаковом входе.

In [ ]:
summary = test_runs.groupby(["target", "model"]).agg(
    parameters=("parameters", "first"), n_test=("n_test", "first"),
    mean_rmse=("test_rmse", "mean"), sd_rmse=("test_rmse", "std"),
)
paired = test_runs.pivot(index=["target", "init_seed", "shuffle_seed"], columns="model", values="test_rmse")
paired["window_mlp_minus_mlp"] = paired.mlp_window6 - paired.mlp
paired["gru_minus_window_mlp"] = paired.gru_window6 - paired.mlp_window6
paired["gru_minus_hypergraph"] = paired.gru_window6 - paired.hypergraph_mgs_kan
display(summary.round(4))
display(paired.round(4))
output.mkdir(parents=True, exist_ok=True)
test_runs.to_csv(output / "runs.csv", index=False)
summary.to_csv(output / "summary.csv")
paired.to_csv(output / "paired.csv")
print(f"Результаты: {output}")